# 自我进化的智能体：自主智能体再训练 Cookbook

## 概述

智能体系统在概念验证之后常常陷入瓶颈期，因为它们依赖人来诊断边界情况并纠正失败。本 cookbook 介绍了一个可重复的再训练循环，它能捕获这些问题、从反馈中学习，并将改进推回到类生产环境的工作流中。我们以一个受监管的医疗文档任务为基础来展示这一方法，但这些模式同样适用于任何要求准确性、可审计性和快速迭代的领域。

### 你将学到什么
- 诊断自主智能体为何达不到生产就绪的标准，并为其加入可衡量的反馈信号。
- 比较三种提示词优化策略——从快速的手动迭代到完全自动化的循环——并了解何时该采用哪一种。
- 组建一个自我修复的工作流，将人工审查、LLM 评审（LLM-as-judge）评估和迭代式提示词改进结合起来。

### 本 notebook 适合谁
- 需要超越玩具演示的 ML/AI 工程师和解决方案架构师。
- 希望获得可执行产出物、以便将其改造成内部工具或生产流水线的产品和交付团队。

### 如何学习本 notebook
1. 从第 1 节开始，了解医疗用例、基线智能体和系统架构。
2. 通过第 2 节，在 OpenAI Evals 界面中练习提示词优化并收集结构化反馈。
3. 运行第 3 节，借助评分器、评估和再训练逻辑实现优化循环的自动化。
4. 在根据你的环境调整工作流时，参考附录中可复用的提示词、配置和评估模板。

本 notebook 采用模块化设计——在将再训练循环应用到你自己的智能体时，你可以独立运行各节，也可以按顺序运行。

## 1. 用例概述：医疗领域中的自我进化智能体

### 问题定义

在本 cookbook 中，我们聚焦于一个**真实世界的用例**：为制药公司起草监管文件。这些机构必须准备大量文件并提交给监管机构（例如美国食品药品监督管理局 FDA），才能获得新药批准。这些申报材料的准确性和速度至关重要，因为它们直接影响救命疗法能以多快的速度送达患者手中。

起草监管文件是一个高度复杂、反复迭代且对精确性要求极高的过程，需要深厚的科学、医学和合规专业知识。尽管已有先进的写作工具，这项工作仍然耗费大量人力，而且容易出现人为错误。**智能体系统可以提供巨大的助力**，协助完成研究综合、内容生成和文档结构化，但仍需要人类专家来确保事实准确性和法规合规性。

关键挑战在于设计一个反馈循环，使这些智能体系统能够迭代学习，并随时间推移不断改进模型行为。这样的系统可以逐步把人的精力从细致的纠错转移到高层次的监督上，在保持监管申报所需严格标准的同时提升效率。

### 自我进化的智能体

下图展示了通过反馈、元提示（meta prompting）和评估持续改进 AI 智能体的迭代过程。该循环结合人工判断或使用 LLM 评审的自动化反馈，迭代地提升性能。

<img src="../../../images/baseline_agent.png" alt="自我进化循环" style="max-width:50%"/>
<br><em>图 1 - 展示用于自动改进智能体的自我进化循环的示意图。</em>

该过程包括以下步骤：

1. **基线智能体**  
   该过程从一个基线智能体开始。在本 notebook 中，我们使用一个刻意简化的示例（一个对文档各节进行摘要的智能体）来说明迭代改进循环。在真实世界或企业环境中，基线智能体可能会复杂得多。它生成的摘要将作为后续评估和改进的初始基准。

2. **人工反馈（或 LLM 评审）**  
   接下来，基线智能体的输出会由人工审查者（例如在生产环境中）和/或自动化的 **LLM 评审**系统进行评估。这一步会收集定量和定性反馈，表明智能体在多大程度上达成了目标——例如，如果我们测试的是摘要长度，反馈可能是“摘要太长”，也可能是评估在判断摘要是否少于 500 词时生成的数值分数（通常介于 `0` 和 `1` 之间）。

3. **评估与汇总分数**  
   根据收集到的反馈生成新的提示词，并通过评估（**Evals**）对其进行测试。这些测试依据预定义的标准衡量性能，并将结果合并为一个反映整体表现的汇总分数。循环会一直进行，直到分数超过目标阈值（例如 `0.8`），或达到最大重试次数（例如 `max_retry = 10`）。如果达到重试上限，系统会提醒工程师需要进行人工改进。

4. **更新后的基线智能体**  
   一旦某个改进版本达到目标性能，它就会取代原来的基线智能体。这个更新后的智能体将成为下一轮迭代的基础，支撑学习、反馈和优化的持续循环。



### 数据集概述

用于评估的数据集包含约 70 个章节，摘自 _Sample CMC Section for Hyperpolarized Pyruvate (13C) Injection_，可在[这里](https://dctd.cancer.gov/drug-discovery-development/reagents-materials/imaging-ind-resources/documentation/13c-pyruvate-cmc.pdf)公开获取。该数据集提供了真实的领域专业内容，适合同时测试科学摘要能力和法规合规行为。


### 基线智能体概述

为了让本 cookbook 保持自包含且易于复现，我们在保留其核心复杂性的同时，对监管文件起草用例进行了简化。在生产环境中，典型的监管文件写作智能体由多个专门的子智能体组成，分别负责起草、数据分析、合规检查、引用生成和事实核查等任务。

在本指南中，我们缩小了监管文件写作智能体的范围，以聚焦于系统的自我修复能力。我们的监管文件写作智能体由两个子智能体组成：
- **摘要器**：生成科学而简洁的摘要。
- **合规检查器**：根据关键法规要求（例如 FDA 21 CFR Part 11）评估每份摘要。

<img src="../../../images/simplified_reg_agent.png" alt="基线智能体" style="max-width:50%"/>
<br><em>图 2 - 在 AgentBuilder UI 中创建的基线智能体。</em>

在本 cookbook 的其余部分，我们实现了摘要器智能体的一个简化版本（参见下文 **智能体设置** 一节）。你也可以复用通过 AgentBuilder 创建的智能体代码。如果你想直接在 AgentBuilder UI 中复现该智能体，以下是所用的关键提示词和参数：

- **摘要器智能体：** 该智能体使用了文件搜索工具，并将 [CMC PDF](data/c13_pyruvate_sample_CMC_from_UCSF.pdf) 上传到了向量存储中。
> _提示词：_ "Summarize section {{workflow.input_as_text}} from {{state.cmc_pdf}} uploaded to the vector store."

- **合规检查器智能体：**
> _提示词：_ "Verify that the summary below is compliant with FDA 21 CFR Part 11: {{input.output_text}}. If the summary is compliant, return _Compliant_. Otherwise, return _This section needs to be manually summarized_."

两个智能体都使用默认参数配置——使用 GPT-5、低推理强度，并以文本作为输出格式。

### 评估方法

评估基线智能体主要有两种方法：

1. **收集人工反馈。** 这种方法通过 OpenAI Evals 平台（或为特定应用构建的自定义 UI）收集人类用户的反馈。它最适合生产环境，或在领域专家（SME）于真实场景中使用工具的试点阶段。这种方法有助于发现开发过程中可能未被识别的边界情况。在 Evals 平台上，用户可以给出点赞或点踩评分，并分享关于摘要的定性反馈。


2. **使用 LLM 评审。** 这种方式通常用于开发阶段，无需占用领域专家的时间就能实现快速的反馈循环。**LLM 评审**使用 LLM 根据预定义的标准自动评估智能体的输出并打分。它也可以用于监测模型漂移（例如在生产环境中），或验证不同模型及模型版本之间的变化（例如在 `gpt-5` 和 `gpt-5-mini` 之间切换）。


本 cookbook 将演示这两种方法：
- **第 2 节**展示通过平台 UI 手动优化提示词的方法
- **第 3 节**使用 LLM 评审实现完全自动化的 API 方法

_注意：Evals 平台目前尚未提供以编程方式获取用户反馈的 API。_


## 2. 使用 OpenAI Evals 平台

OpenAI Evals 平台为提示词优化和评估提供了直观的界面。本节演示从上传数据集到迭代改进提示词的完整工作流，展示如何在实施自动化方案之前，利用平台的可视化界面来优化提示词。

### 第 1 步：上传数据集

要开始使用 OpenAI 评估平台，首先需要上传数据集：

1. 点击 **+ Create** 按钮
2. 定义数据集名称
3. 上传 CSV 文件并选择要保留的列
4. 上传

数据集应包含需要摘要的文档或文档章节。每一行代表系统要处理的一条输入。

### 第 2 步：浏览数据

上传完成后，你可以浏览数据集。点击数据集名称即可查看上传的数据。这样可以在配置提示词之前，确认数据格式正确且包含预期的内容。

### 第 3 步：配置初始提示词

在这一步，你需要定义初始系统提示词，并配置数据如何流经模型。

<img src="../../../images/prompt_input.png" alt="平台提示词配置" style="max-width:50%">
<br><em>图 3 - 平台的 "New prompt" 界面，显示模型配置、变量和系统消息设置。</em>


#### 配置步骤

1. **系统提示词**：添加定义模型任务和行为的系统消息（该提示词将被优化）
2. **用户提示词模板**：为用户消息添加提示词模板，使用诸如 `{{<column_name>}}` 之类的变量，这些变量会被替换为数据集中的实际数据
3. **模型选择**：选择用于生成的模型（例如 gpt-4.1、gpt-5）
4. **Temperature**：在创造性与确定性之间进行配置

你可以从一个非常简单的提示词开始，以展示优化过程的威力。例如，仅从 "summarize" 开始，就能展示系统如何从一个极简的起点逐步进化。

### 第 4 步：生成输出

提示词配置完成后，就可以在整个数据集上生成输出了。提示词会对每一行运行一次，输出将生成在一个新的 **output** 列中。

1. 点击 **"Generate Output"**
2. 平台会针对所有样本运行你的提示词
3. 结果会出现在新的 **Output** 列中

平台会处理数据集中的每一行，将模板变量替换为实际值，并使用你的系统提示词调用模型。这会生成一组可供评估的基线输出。

### 第 5 步：审查和评估

在评估环节，你需要提供结构化反馈来指导提示词改进。

#### 审查输出

1. **添加评估列**（如果没有自动添加）——点击 "Columns" → "Annotations" → "Add"：
   - **Rating** - 二元（好/坏）或数值评分
   - **Feedback** - 描述需要改进之处的文本

2. **提供评分和反馈** - 为每条输出添加你的评估。

   根据输出的质量，你可以选择好或坏的评分，并结合你希望如何改进答案来解释你的打分。例如：

      > （评分）| 反馈
      > - （好）不错，但只应提供答案。输出不应包含标题或答案以外的任何文本。
      > - （坏）信息不错，但应以项目符号的形式呈现。
      > - （好）摘要很好，清晰明了。
      > - （坏）回答时使用项目符号以提高可读性。对每个小节分别进行摘要。

3. **保存标注** - 你的反馈会随评估运行一起保存

<img src="../../../images/feedback.png" alt="平台评估界面" style="max-width:50%">
<br><em>图 4 - 评估界面，显示生成的输出以及用于标注的评分和反馈列。</em>

这些结构化反馈将成为自动优化提示词的基础。

### 第 6 步：优化提示词

收集反馈后，平台可以自动生成改进后的提示词。

1. 点击 **"Optimize"**
2. 新的提示词版本会在一个新选项卡中生成
3. 点击 **"View Prompt"** 查看改进后的版本

<img src="../../../images/updated_prompt.png" alt="平台优化后的提示词" style="max-width:50%">
<br><em>图 5 - 平台生成的改进后提示词，展示了详细的指令和要求。</em>

### 第 7 步：迭代和比较

改进后的提示词准备就绪后，开始新一轮迭代来衡量改进效果。

1. 点击 **"Generate Output"**
2. 审查新结果，并就仍存在的问题提供反馈
3. 如有需要，再次点击 **"Optimize"**
4. 重复上述步骤，直到满意为止

平台的选项卡结构让你可以比较各轮迭代的表现。你可以轻松看到输出是如何从初始提示词演变到优化版本的。

<img src="../../../images/updated_prompt_feedback.png" alt="平台更新后提示词的反馈" style="max-width:50%">
<br><em>图 6 - 优化后提示词的反馈和评估结果，显示输出质量有所提升。</em>

#### 何时停止迭代

持续进行优化循环，直到：
- **达到质量阈值**：超过 80% 的输出获得正面反馈
- **收益递减**：新的迭代只带来极小的改进
- **具体问题已解决**：所有已识别的失败模式都已得到处理

这种基于平台的方法为理解提示词优化打下了很好的基础，之后再转向自动化实现。可视化界面让你能够轻松看到改动的影响，并理解优化过程。


## 3. 使用 LLM 评审的自我进化循环

本节介绍一个完全自动化的评估工作流：通过 OpenAI API 使用 LLM 评审，无需任何用户界面。这种方法能够以可扩展、程序化的方式评估智能体性能，支持在生产环境中快速迭代和持续监控模型。

In [ ]:
# gepa and litellm are only required for the Section 4.b (prompt optimization with GEPA)
%pip install --upgrade openai openai-agents pydantic pandas gepa litellm python-dotenv -qqq 
%load_ext dotenv
%dotenv

# Place your API key in a file called .env
# OPENAI_API_KEY=sk-...


### 创建评估

为了评估基线摘要智能体，我们使用四个互补的评分器，在确定性检查与语义判断之间取得平衡。

| 评分器 | 类型 | 通过阈值 | 检查内容 | 原因 |
|---|---|---:|---|---|
| 化学名称字符串 | `python` | 0.8 | 章节中的精确化学名称是否出现在摘要中。 | 强制保留关键的领域实体，避免摘要遗漏具有化学意义的术语。 |
| 摘要长度 | `python` | 0.85 | 与预期 100 词长度之间偏差的倒数。 | 让摘要保持简洁、可比较，减少可能掩盖内容质量问题的冗长表述。 |
| 余弦相似度 | `text_similarity` | 0.85 | 章节文本与摘要文本之间的余弦相似度。 | 确保摘要紧扣源内容，而不会在语义上偏离。 |
| LLM 评审 | `score_model` | 0.85 | 由充当评估者的模型依据评分标准给出的分数。 | 捕捉基于规则的指标所遗漏的细微质量信号，提升整体稳健性。 |

**说明**
- 两个 Python 评分器能及早发现领域保真度和长度控制方面的问题，从而在语义调优之前让优化过程保持稳定。
- 文本相似度可以防止偏离源内容的表面化改写。
- 当边界情况绕过了确定性检查时，LLM 评审可以提供整体性的兜底保障。

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {"section": {"type": "string"}, "summary": {"type": "string"}},
        "required": ["section", "summary"],
    },
    "include_sample_schema": False,
}

testing_criteria = [
    {
        "type": "python",
        "name": "chemical_name_grader",
        "image_tag": "2025-05-08",
        "pass_threshold": 0.8,
        "source": r"""def grade(sample: dict, item: dict) -> float:
    section = item["section"]
    summary = item["summary"]
    CHEMICALS_MASTER = ["[1-¹³C]Pyruvic acid","[1-¹³C]Pyruvate","¹²C Pyruvic acid","Sodium [1-¹³C]pyruvate","Sodium pyruvate (¹²C)","AH111501 (Trityl radical)","Tris{8-carboxyl-2,2,6,6-tetra[2-(1-methoxyethyl)]-benzo(1,2-d:4,5-d’)bis(1,3)dithiole-4-yl}methyl acid","AH111501 sodium salt","Methyl, tris[8-carboxy-2,2,6,6-tetrakis(2-methoxyethyl)benzo[1,2-d:4,5-d’]bis[1,3]dithiol-4-yl]-, trisodium salt","AH111501 trisodium salt","AH111576","2,2′,2″,2‴-(4,8-Dibromobenzo[1,2-d:4,5-d′]bis([1,3]dithiole)-2,2,6,6-tetrayl)tetraethanol","AH111586","4,8-Dibromo-2,2,6,6-tetrakis(2-methoxyethyl)benzo[1,2-d:4,5-d′]bis([1,3]dithiole)","AH111709","AH111743","AH112615","4,4-Bis-hydroxymethyl-2-methyl-oxazolidine-2-carboxylic acid","AH112623","Parapyruvate","2-Hydroxy-2-methyl-4-oxo-pentanedioic acid","AH113127","(4-Hydroxymethyl-oxazolidin-4-yl)-methanol","AH113462/E","Enol lactone","AH113462/K","Keto lactone","Acetyl bromide","Methanol","Dimethyl sulfoxide","DMSO","Tetrahydrofuran","THF","Acetonitrile","ACN","Diethyl ether","Et₂O","N,N-Dimethylacetamide","DMA","1,3-Dimethyl-2-imidazolidinone","DMI","Hydrochloric acid","HCl","Sodium hydroxide","NaOH","Disodium ethylenediaminetetraacetate","Na₂EDTA","Ethylenediaminetetraacetic acid","EDTA","Tris(hydroxymethyl)aminomethane","TRIS","Trometamol","Trifluoroacetic acid","TFA","Toluene","Heptane","Ethyl acetate","Ethanol","Water","H₂O","Sodium chloride","NaCl","Cuprous [1-¹³C]cyanide","Cu¹³CN","Gadolinium","Gd","Tin","Sn","Phosphorus","P","Carbon dioxide","CO₂","Sodium [1-13C]pyruvate","[1-13C]Pyruvic acid","1-13C pyruvate"]

    # Identify the chemicals present in the section
    present = [chem for chem in CHEMICALS_MASTER if chem in section]

    # If no chemicals present, consider it satisfied
    if not present:
        return 1.0

    correct = 0
    for chem in present:
        # Only count as correct if the exact chemical string appears in the summary
        if chem in summary:
            correct += 1

    return correct / len(present)""",
    },
    {
    "type": "python",
    "name": "word_length_deviation_grader",
    "image_tag": "2025-05-08",
    "pass_threshold": 0.85,
    "source": r"""
def grade(sample: dict, item: dict) -> float:
    summary = item["summary"]
    word_count = len(summary.split())
    
    expected_summary_length = 100
    tolerance = 0.2  # 20% band around target
    
    # relative deviation
    deviation = abs(word_count - expected_summary_length) / expected_summary_length
    
    # If within tolerance band → full score
    if deviation <= tolerance:
        return 1.0
    
    # Outside band → score decays linearly, capped at 0
    # e.g., deviation 0.3 → score 0.8, deviation 1.0+ → 0.0
    score = 1.0 - (deviation - tolerance)
    return max(0.0, score)
""",
},
    {
        "name": "cosine_similarity",
        "type": "text_similarity",
        "input": "{{ item.summary }}",
        "reference": "{{ item.section }}",
        "evaluation_metric": "cosine",
        "pass_threshold": 0.85,
    },
    {
        "name": "llm_as_judge",
        "type": "score_model",
        "model": "gpt-4.1",
        "input": [
            {
                "role": "system",
                "content": (
                    "You are an expert technical summarization evaluator. "
                    "Evaluate whether the summary captures and preserves the important technical facts and specific details from the section, allowing for occasional minor rewording or omissions of less important points, but not major technical inaccuracies or information loss.\n\n"
                    "Scoring Guidelines:\n"
                    "- Return a numerical score between 0 and 1 (with up to two decimal places).\n"
                    "- A score of 1 means the summary is almost flawless: it is comprehensive, highly faithful, and technically accurate, with virtually no important or meaningful details missing, and no significant misstatements or distortions.\n"
                    "- 0.75-0.99 indicates excellent work: all main facts are represented, but there may be trivial omissions or very minor rewording that do not materially affect understanding.\n"
                    "- 0.5-0.75 indicates good but imperfect: most technical information is retained and correctly presented, some less critical details might be missing or slightly rephrased, but overall fidelity is preserved.\n"
                    "- 0.3-0.5 means significant information is missing, or some technical inaccuracies are present, but the summary retains a reasonable portion of key facts.\n"
                    "- 0.0-0.3 means there are major omissions, misunderstandings, or a failure to capture the most important technical content.\n\n"
                    "Respond only with a single number between 0 and 1 indicating summary quality by these criteria."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Section:\n{{item.section}}\n"
                    "Summary:\n{{sample.output_text}}"
                ),
            },
        ],
        "range": [0, 1],
        "pass_threshold": 0.85,
    },
]

eval = client.evals.create(
    name="self_evolving_eval",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"Created Eval: {eval.id}")

你应该会在输出中看到一个评估 ID，例如 `eval_...`。这就是我们刚刚创建的评估的 ID（如下所示）

<img src="../../../images/eval_set_config.png" alt="平台评估配置" style="max-width:50%">
<br><em>图 7 - 平台的 Eval 界面，显示数据源配置和测试标准设置。</em>

### 评分器打分与解析

接下来，我们需要针对摘要智能体的输出运行评估，并解析结果以获得评估中各评分器的分数。为此，我们将使用几个辅助函数：
- `run_eval`：一个简单的运行器，以正确的格式调用 evals API
- `poll_eval_run`：一个轮询工具，用于等待已调度的评估运行完成
- `parse_eval_run_output`：解析评估运行，并为反馈循环返回结构化输出

In [ ]:
import time
import json

def run_eval(eval_id: str, section: str, summary: str):
  """Creates a run of the eval with the input section and output summary."""
  return client.evals.runs.create(
    eval_id=eval_id,
    name="self-evolving-eval",
    data_source={
      "type": "jsonl",
      "source": {
        "type": "file_content",
        "content": [
          {
            "item": {
              "section": section,
              "summary": summary,
            }
          }
        ],
      },
    },
  )


def poll_eval_run(eval_id: str, run_id: str, max_polls = 10):
    """
    Polls the evaluation run until completion or timeout.

    This function exists to handle asynchronous behavior in the eval service by
    periodically checking run status. It balances responsiveness and resource use by
    polling at fixed intervals rather than blocking indefinitely. The retry limit
    prevents runaway loops in cases where the service never returns a completed status.
    """
    run = None
    for attempt in range(1, max_polls + 1):
        run = client.evals.runs.retrieve(eval_id=eval_id, run_id=run_id)
        if run.status == "completed":
            break
        if attempt == max_polls:
            print("Exceeded retries, aborting")
            break

        time.sleep(5)

    run_output_items = client.evals.runs.output_items.list(
        eval_id=eval_id, run_id=run_id
    )
    return run_output_items


def parse_eval_run_output(items):
    """Extract all grader scores and any available conclusion outputs."""
    all_results = []

    for item in items.data:
        for result in item.results:
            grader_name_full = result.name
            score = result.score
            passed = result.passed
            reasoning = None
            try:
                sample = result.sample
                if sample:
                    content = result.sample["output"][0]["content"]
                    content_json = json.loads(content)
                    steps = content_json["steps"]
                    reasoning = " ".join([step["conclusion"] for step in steps])
            except Exception:
                pass

            all_results.append(
                {
                    "grader_name": grader_name_full,
                    "score": score,
                    "passed": passed,
                    "reasoning": reasoning,
                }
            )

    return all_results

现在，我们可以使用之前创建的评估 ID，针对任意输入章节和摘要输出运行评分器。这构成了反馈循环的核心，它将启动提示词优化流程。

### 评估执行运行

我们直接提供一个章节和一份生成的摘要来测试评估。

In [ ]:
EVAL_ID = eval.id #Created eval ID from above cell
SECTION = "3.2.S.1 General Information ([1-13C]pyruvic acid) The active ingredient in Hyperpolarized Pyruvate (13C) Injection is hyperpolarized [1-13C]pyruvate. The drug substance is defined as [13C]pyruvic acid, which is neutralized to [1-13C]pyruvate during the compounding process. In several pre-clinical and clinical studies and during evaluation of stability, pyruvic acid has been used instead of [1-13C]pyruvic acid (see Sections 3.2.P.2.2.1 Formulation Development for Hyperpolarized Pyruvate (13C) Injection and Section 8.1 Introduction for Item 8 Pharmacology and Toxicology Info). In the Section 3.2.S Drug Substance, data are presented for both pyruvic acid and for [1-13C]pyruvic acid. For simplicity, the terminology used in headings and captions is [1-13C]pyruvic acid. Batches containing pyruvic acid are specified by footnotes. 3.2.S.1.1 Nomenclature ([1-13C]pyruvic acid) The drug substance used for compounding of Hyperpolarized Pyruvate (13C) Injection is [1-13C]pyruvic acid. Company code: W6578 Chemical name: [1-13C]pyruvic acid CAS registry number: 127-17-3 3.2.S.1.2 Structure ([1-13C]pyruvic acid) Figure 1 Structure of [1-13C]pyruvic acid Molecular formula: C H O 3 4 3 Molecular weight: 89.06 3.2.S.1.3 General Properties ([1-13C]pyruvic acid) Appearance: Colorless to yellow, clear, viscous liquid pKa:Ka:aranWater solubility: Complete The structure of [1-13C]pyruvic acid has been confirmed by spectroscopic analysis (see Section 3.2.S.3.1 Elucidation of Structure and other Characteristics)."
SUMMARY = "The active ingredient in Hyperpolarized Pyruvate (13C) Injection is hyperpolarized [1-13C]pyruvate, derived from [1-13C]pyruvic acid (neutralized during compounding). Both pyruvic acid and [1-13C]pyruvic acid were used in studies and stability evaluations, but the documentation refers to [1-13C]pyruvic acid unless otherwise noted. The drug substance ([1-13C]pyruvic acid, CAS 127-17-3) is a colorless to yellow, clear, viscous liquid with a molecular formula C3H4O3 and molecular weight 89.06. Its structure has been confirmed by spectroscopic analysis, and it is completely soluble in water."

eval_run = run_eval(EVAL_ID, section=SECTION, summary=SUMMARY)
run_output = poll_eval_run(eval_id=EVAL_ID, run_id=eval_run.id)

grader_scores = parse_eval_run_output(run_output)
print(grader_scores)

你应该会在输出中看到一组评分器分数，例如：

```[{'grader_name': 'chemical_name_grader-<uuid>', 'score': 0.5, 'passed': False, 'reasoning': None}, {'grader_name': 'word_length_deviation_grader-<uuid>', 'score': 0.8, 'passed': True, 'reasoning': None}, {'grader_name': 'cosine_similarity-<uuid>', 'score': 0.9104484223477793, 'passed': True, 'reasoning': None}, {'grader_name': 'llm_as_judge-<uuid>', 'score': 0.8, 'passed': True, 'reasoning': 'The summary needs to include specific details from the section. Part of the essential information is captured. Key pieces of information are missing. Not all relevant structural information is included.'}]```


运行这个脚本后可以看到，除了 `chemical_name_grader` 之外，大多数评分器都通过了。接下来，我们将以编程方式识别出这一改进摘要智能体的机会。

_注意：在本地运行时，除 `chemical_name_grader` 之外的其他评分器一开始也可能不通过。这很正常，评分器最初可能会失败，但结果应会通过反馈循环得到改善。早期的失败只是反映了模型在收敛到更准确的结果之前对其回答所做的调整。_


### 仪表盘可观测性
评估运行和结果也可以在 OpenAI Dashboard 中查看：

<img src="../../../images/eval_dashboard.png" alt="评估仪表盘" style="max-width:50%">
<br><em>图 8 - 显示评估运行和结果的评估仪表盘。</em>


我们还可以深入查看某次具体的评估运行：
<img src="../../../images/eval_run_results.png" alt="评估结果" style="max-width:50%">
<br><em>图 9 - 详细的评估运行结果，显示评分器分数和性能指标。</em>


## 智能体设置

评估和评分器设置好之后，我们就可以回到摘要智能体了。
为简单起见，我们在下面提供一个简单智能体的代码。你也可以像图 2 所示那样使用 `AgentBuilder`，并从 UI 中导出代码。


我们还需要一个元提示优化智能体来优化提示词，以及一些用于处理提示词版本的简单工具：
- `PromptVersionEntry`：一个 pydantic 模型，用于跟踪提示词及其元数据在生产环境中的变化
- `VersionedPrompt`：一个用于跟踪提示词版本的工具类。在生产环境中，它对于分析提示词的演变过程，以及在出现回退时确保有可回滚的历史版本都非常重要

In [ ]:
from datetime import datetime
from typing import Any, Optional

from pydantic import BaseModel, Field, ConfigDict, field_validator

class PromptVersionEntry(BaseModel):
    """Data model for a prompt and associated data for observability"""
    version: int = Field(
        ..., ge=0, description="Version number of the prompt (increments)"
    )
    model: str = Field(
        "gpt-5",
        min_length=1,
        description="The model version to use for this version of the prompt, defaults to gpt-5",
    )
    prompt: str = Field(
        ..., min_length=1, description="The prompt text for this version"
    )
    timestamp: datetime = Field(
        default_factory=datetime.utcnow,
        description="UTC timestamp when this version was created",
    )
    eval_id: Optional[str] = Field(
        None, description="ID of the evaluation associated with this prompt version"
    )
    run_id: Optional[str] = Field(
        None, description="ID of the run associated with this prompt version"
    )
    metadata: Optional[dict[str, Any]] = Field(
        None, description="Free-form metadata dict (e.g., section, summary)"
    )

    model_config = ConfigDict(
        str_strip_whitespace=True, validate_assignment=True, extra="forbid"
    )

    @field_validator("prompt")
    @classmethod
    def prompt_not_blank(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("prompt must not be blank or only whitespace")
        return v


class VersionedPrompt:
    """Manages a collection of prompt versions and provides controlled updates and rollbacks."""
    def __init__(
        self,
        initial_prompt: str,
        model: Optional[str] = "gpt-5",
        eval_id: Optional[str] = None,
        run_id: Optional[str] = None,
        metadata: Optional[dict[str, Any]] = None,
    ):
        if not initial_prompt or not initial_prompt.strip():
            raise ValueError("initial_prompt must be non-empty")
        self._versions: list[PromptVersionEntry] = []
        first_entry = PromptVersionEntry(
            version=0,
            prompt=initial_prompt,
            model=model,
            eval_id=eval_id,
            run_id=run_id,
            metadata=metadata,
        )
        self._versions.append(first_entry)

    def update(
        self,
        new_prompt: str,
        model: Optional[str] = "gpt-5",
        eval_id: Optional[str] = None,
        run_id: Optional[str] = None,
        metadata: Optional[dict[str, Any]] = None,
    ) -> PromptVersionEntry:
        if not new_prompt or not new_prompt.strip():
            raise ValueError("new_prompt must be non-empty")

        version = self.current().version + 1
        entry = PromptVersionEntry(
            version=version,
            prompt=new_prompt,
            model=model,
            eval_id=eval_id,
            run_id=run_id,
            metadata=metadata,
        )
        self._versions.append(entry)
        return entry

    def current(self) -> PromptVersionEntry:
        return self._versions[-1]

    def revert_to_version(self, version: int) -> PromptVersionEntry:
        idx = None
        for i, entry in enumerate(self._versions):
            if entry.version == version:
                idx = i
                break

        if idx is None:
            raise ValueError(f"No version found with version={version}")

        self._versions = self._versions[: idx + 1]
        return self._versions[-1]

接下来，我们创建初始的摘要智能体和提示词优化智能体。

_注意：由于摘要智能体的提示词预计会在生产环境中不断演进，我们为它创建了一个包装器来跟踪提示词的变化；而在本 cookbook 中，元提示智能体的提示词将保持不变。_

In [ ]:

from agents import Agent

METAPROMPT_TEMPLATE = """
# Context:
## Original prompt:
{original_prompt}

## Section:
{section}

## Summary:
{summary}

## Reason to improve the prompt:
{reasoning}

# Task:
Write a new summarization prompt that is significantly improved and more specific than the original.  
The new prompt should instruct the model to produce concise yet comprehensive technical summaries that precisely preserve all explicit information from the source text. It should emphasize the inclusion of all named entities, quantities, compounds, and technical terminology without paraphrasing or omission. The resulting prompt should read like a clear, directive system message for a technical summarization assistant—structured, unambiguous, and generalizable across scientific or regulatory document sections.
"""

metaprompt_agent = Agent(
    name="MetapromptAgent", instructions="You are a prompt optimizer."
)

summarization_prompt = VersionedPrompt(
    initial_prompt="""You are a summarization assistant.
Given a section of text, produce a summary."""
)

def make_summarization_agent(prompt_entry: PromptVersionEntry) -> Agent:
    return Agent(
        name="SummarizationAgent",
        instructions=prompt_entry.prompt,
        model=prompt_entry.model,
    )

summarization_agent = make_summarization_agent(summarization_prompt.current())

# Cache eval results by section + summary so repeated attempts do not trigger redundant grader runs.
eval_cache: dict[tuple[str, str], list[dict[str, Any]]] = {}

# Track the highest-scoring candidate that also passes the lenient score threshold.
best_candidate: dict[str, Any] = {
    "score": float("-inf"),
    "prompt": summarization_prompt.current().prompt,
    "model": summarization_prompt.current().model,
    "summary": None,
    "metadata": None,
    "version": summarization_prompt.current().version,
    "passed_lenient": False,
    "total_score": float("-inf"),
}

# Aggregate per-version performance so we can pick the strongest total scorer at the end.
aggregate_prompt_stats: dict[int, dict[str, Any]] = {}



### 编排与监控

到目前为止，我们已经创建了：
- 包含 4 个评分器的评估，用于评估输出并为每个评分器给出分数
- 一个摘要智能体，配有带版本控制的提示词类，用于跟踪提示词和模型的变化
- 一个元提示优化智能体，它会尝试根据一组推理来更新提示词

现在，可以将这些不同的功能组合起来，编排自我进化循环，并在 OpenAI 仪表盘中进行智能体追踪。

请记住，这是一个简化的示例。在真实场景中，你需要确保为优化尝试设置护栏，并在护栏被触发时通过告警通知相关人员。

_注意：受限于 cookbook 的实际条件，我们通过输入一个静态数据集来模拟数据流，并使用 `print` 语句代替真正的可观测性方案。_

### 编排工具

与前面几节一样，我们会创建一些工具来管理反馈循环的编排逻辑。

In [ ]:
import asyncio
from typing import Any, Optional
from agents import Runner

LENIENT_PASS_RATIO = 0.75 # 75% of graders must pass (binary) 
LENIENT_AVERAGE_THRESHOLD = 0.85 # 85% average score across graders 

def reset_best_candidate() -> None:
    """Reset the best candidate tracker for a new optimization run."""
    global best_candidate

    current = summarization_prompt.current()
    best_candidate = {
        "score": float("-inf"),
        "prompt": current.prompt,
        "model": current.model,
        "summary": None,
        "metadata": None,
        "version": current.version,
    }

def reset_best_trackers() -> None:
    """Reset both the best-candidate tracker and aggregate stats."""
    reset_best_candidate()
    aggregate_prompt_stats.clear()


def update_best_candidate(
    *,
    average_score: Optional[float] = None,
    prompt_text: str,
    model_name: str,
    summary_text: str = None,
    metadata: dict[str, Any] = None,
    lenient_passed: bool = False,
    prompt_version: int = None,
    total_score: Optional[float] = None,
    score: Optional[float] = None,
) -> None:
    """Persist the best lenient-passing candidate."""
    global best_candidate

    if prompt_version is None:
        prompt_version = summarization_prompt.current().version

    if average_score is None:
        average_score = score

    if average_score is None:
        return

    if lenient_passed:
        best_candidate.update(
            {
                "score": average_score,
                "prompt": prompt_text,
                "model": model_name,
                "summary": summary_text,
                "metadata": metadata,
                "version": prompt_version,
                "total_score": total_score if total_score is not None else average_score,
            }
        )


def apply_best_candidate_if_needed() -> Agent:
    """Ensure summarization_prompt reflects the best prompt candidate."""
    if best_candidate["score"] > float("-inf"):
        current = summarization_prompt.current()
        target = best_candidate
        # Only update if different
        if (
            current.prompt != target["prompt"]
            or current.model != target["model"]
            or current.version != target.get("version")
        ):
            summarization_prompt.update(
                new_prompt=target["prompt"],
                model=target["model"],
                metadata=target.get("metadata"),
            )
            target["version"] = summarization_prompt.current().version
        return make_summarization_agent(summarization_prompt.current())

    return make_summarization_agent(summarization_prompt.current())


def record_aggregate_prompt_score(
    *,
    prompt_version: int,
    prompt_text: str,
    model_name: str,
    average_score: float,
    total_score: Optional[float] = None,
) -> None:
    """Accumulate per-version grader scores for aggregate selection."""
    stats = aggregate_prompt_stats.setdefault(
        prompt_version,
        {
            "version": prompt_version,
            "prompt": prompt_text,
            "model": model_name,
            "total_score": 0.0,
            "total_average": 0.0,
            "count": 0,
        },
    )
    stats["total_score"] += total_score if total_score is not None else average_score
    stats["total_average"] += average_score
    stats["count"] += 1
    stats["prompt"] = prompt_text
    stats["model"] = model_name


def select_best_aggregate_prompt() -> Optional[dict[str, Any]]:
    """Return the prompt version with the highest cumulative score."""
    if not aggregate_prompt_stats:
        return None
    return max(
        aggregate_prompt_stats.values(),
        key=lambda entry: (
            entry.get("total_score", float("-inf")),
            entry.get("version", -1),
        ),
    )


async def get_eval_grader_score(eval_id: str, section: str, summary: str):
    """Retrieve grader scores for a section-summary pair with caching."""
    cache_key = (section, summary)
    if cache_key in eval_cache:
        return eval_cache[cache_key]

    eval_run = run_eval(eval_id=eval_id, section=section, summary=summary)
    run_output = poll_eval_run(eval_id=eval_id, run_id=eval_run.id)
    results = parse_eval_run_output(run_output)
    eval_cache[cache_key] = results
    return results


def calculate_grader_score(grader_scores):
    """Simple average score of all graders from the eval."""
    if not grader_scores:
        return 0.0

    score_sum = 0.0
    for entry in grader_scores:
        score_sum += entry.get("score", 0.0)

    return score_sum / len(grader_scores)



def calculate_total_grader_score(grader_scores):
    """Sum of all grader scores for aggregate tracking."""
    if not grader_scores:
        return 0.0

    return sum(entry.get("score", 0.0) for entry in grader_scores)


DEFAULT_PASSING_FEEDBACK = (
    "All graders passed; tighten factual coverage, chemical completeness, and conciseness."
)


def is_lenient_pass(grader_scores, average_score: float) -> bool:
    if not grader_scores:
        return False

    passed_count = sum(1 for entry in grader_scores if entry.get("passed"))
    total_graders = len(grader_scores)

    if total_graders and (passed_count / total_graders) >= LENIENT_PASS_RATIO:
        return True
    return average_score >= LENIENT_AVERAGE_THRESHOLD


def collect_grader_feedback(grader_scores):
    """Consolidate grader reasoning into actionable feedback for the metaprompt agent."""
    feedback_lines = []

    for entry in grader_scores:
        grader = entry.get("grader_name", "")
        passed = entry.get("passed", False)
        reasoning = entry.get("reasoning")

        if not passed:
            if grader.startswith("chemical_name_grader"):
                feedback_lines.append(
                    "Not all chemical names in the input section were included in the summary."
                )
            elif grader.startswith("word_length_deviation_grader"):
                feedback_lines.append(
                    "The summary length deviates too much from the expected length."
                )
            elif grader.startswith("cosine_similarity"):
                feedback_lines.append(
                    "The summary is not sufficiently similar to the source section (cosine similarity too low)."
                )
            elif grader.startswith("llm_as_judge") and reasoning:
                feedback_lines.append(reasoning)

    if not feedback_lines:
        feedback_lines.append(DEFAULT_PASSING_FEEDBACK)

    return "".join(feedback_lines)



### 自我进化循环

现在，为了模拟一连串的摘要请求，我们将输入一个准备好的数据集，观察优化过程如何从一个朴素的提示词开始逐步进化。

> 所引用的 dataset.csv 可以在 Github 仓库中找到。

In [ ]:
import pandas as pd

from agents import Agent, trace

EVAL_ID = eval.id #Created eval ID from above cell
MAX_OPTIMIZATION_RETRIES = 3

async def self_evolving_loop(summarization_agent: Agent) -> Agent:
    print(f"Starting self-evolving loop | Initial prompt v{summarization_prompt.current().version}")
    print(f"Prompt:{summarization_prompt.current().prompt}")
    print("-" * 80)

    reset_best_trackers()
    df = pd.read_csv("data/dataset.csv")

    with trace("Self-evolving Optimization Workflow"):
        for _, row in df.head().iterrows():
            content = row.get("content")
            if pd.isna(content) or (isinstance(content, str) and not content.strip()):
                continue

            section_number = str(row["section_number"])
            section = str(content)
            current_version = summarization_prompt.current().version

            print(f"[Section {section_number}] Using prompt v{current_version}")

            optimization_success = False

            for attempt in range(1, MAX_OPTIMIZATION_RETRIES + 1):
                print(f"  Attempt {attempt}: evaluating summary...")

                summary_result = await Runner.run(summarization_agent, section)
                summary = summary_result.final_output

                grader_scores = await get_eval_grader_score(eval_id=EVAL_ID, summary=summary, section=section)
                average_score = calculate_grader_score(grader_scores)
                total_score = calculate_total_grader_score(grader_scores)
                lenient_passed = is_lenient_pass(grader_scores, average_score)
                print(
                    f"	Scores — avg={average_score:.3f}, total={total_score:.3f}, lenient_passed={lenient_passed}"
                )

                record_aggregate_prompt_score(
                    prompt_version=summarization_prompt.current().version,
                    prompt_text=summarization_prompt.current().prompt,
                    model_name=summarization_prompt.current().model,
                    average_score=average_score,
                    total_score=total_score,
                )

                update_best_candidate(
                    average_score=average_score,
                    prompt_text=summarization_prompt.current().prompt,
                    model_name=summarization_prompt.current().model,
                    summary_text=summary,
                    metadata={
                        "section": section_number,
                        "average_score": average_score,
                        "grader_results": grader_scores,
                        "prompt_version": summarization_prompt.current().version,
                    },
                    lenient_passed=lenient_passed,
                    prompt_version=summarization_prompt.current().version,
                )

                if lenient_passed:
                    optimization_success = True
                    print(f"	Passed with prompt v{summarization_prompt.current().version}")
                    break

                print("	Failed eval. Improving prompt...")
                eval_feedback = collect_grader_feedback(grader_scores)

                metaprompt_result = await Runner.run(
                    metaprompt_agent,
                    input=METAPROMPT_TEMPLATE.format(
                        original_prompt=summarization_prompt.current().prompt,
                        section=section,
                        summary=summary,
                        reasoning=eval_feedback,
                    ),
                )
                improved_prompt = metaprompt_result.final_output
                summarization_prompt.update(
                    new_prompt=improved_prompt,
                    metadata={"section": section, "summary": summary},
                )
                summarization_agent = make_summarization_agent(summarization_prompt.current())

                print(f"	Prompt improved → v{summarization_prompt.current().version}")

            if not optimization_success:
                print(
                    "	All attempts failed; keeping latest prompt version "
                    f"v{summarization_prompt.current().version} for the next section."
                )

    summarization_agent = apply_best_candidate_if_needed()

    print("" + "-" * 80)
    print("Completed optimization loop.")
    print(f"Final prompt version: v{summarization_prompt.current().version}")
    if best_candidate["score"] > float("-inf"):
        print(
            f"Best lenient prompt: v{best_candidate.get('version')} (avg={best_candidate['score']:.3f})"
        )

    aggregate_best = select_best_aggregate_prompt()
    if aggregate_best:
        per_section = (
            aggregate_best.get("total_average", 0.0) / aggregate_best.get("count", 1)
            if aggregate_best.get("count")
            else 0.0
        )
        print(
            f"Aggregate best prompt: v{aggregate_best.get('version')} "
            f"(total={aggregate_best.get('total_score', 0.0):.3f}, avg/section={per_section:.3f}, model={aggregate_best.get('model', 'unknown')})"
        )

    print(f"Final prompt:{summarization_prompt.current().prompt}")
    return summarization_agent

summarization_agent = await self_evolving_loop(summarization_agent)


**最终提示词是如何选出的**

- 每次评估都会记录评分器平均分、所有评分器的总分，以及该次尝试是否通过了宽松标准。
- `best_candidate` 跟踪最近一次通过宽松标准的结果（以保证透明度），但最终的选择使用汇总总分，以确保保留整体表现最好的提示词。
- 循环结束时，`apply_best_candidate_if_needed` 会恢复累计评分器分数最高的提示词（分数相同时优先选择最新版本），从而保证最终呈现的提示词是观察到的表现最强的那一个。


下面是上述代码的一个示例（节选）输出。

检查输出可以看到，自我进化的提示词确实奏效了。有几点值得注意：
1. 优化并不总是成功，因此能够回滚提示词版本非常重要
2. 评分器所提供信息的保真度，对于确保优化质量至关重要

### 智能体日志与追踪

我们可以在仪表盘的日志中查看优化工作流的运行情况：

<img src="../../../images/agent_log_traces.png" alt="智能体日志追踪" style="max-width:50%">
<br><em>图 10 - 智能体日志追踪，显示仪表盘中的优化工作流运行。</em>

还可以深入查看各个智能体调用：

<img src="../../../images/agent_trace_details.png" alt="智能体追踪详情" style="max-width:50%">
<br><em>图 11 - 详细的智能体追踪，显示各个智能体调用和执行流程。</em>

### 持续监控

评估循环完成后，系统应继续监控新流入的数据，并定期在盲测数据集上重新评估模型性能。这可以确保在数据分布不断变化的情况下，模型依然保持准确和合规。

要实现持续监控，你可以集成一个 cron 任务或轻量级的调度循环，定期检查数据源中的更新（例如新上传的 PDF 或新的数据库条目）。一旦检测到新数据，系统就会自动触发前面介绍的评估和优化循环。

例如（伪代码）：

In [ ]:
# this cell is pseudo-code and not meant to be run as-is

import time

def continuous_monitoring(interval_hours=24):
    """Periodically check for new data and trigger the evaluation loop."""
    while True:
        print("Checking for new data...")
        if new_data_detected():
            print("New data found — running evaluation and optimization loop.")
            self_evolving_loop()
        else:
            print("No new data. Sleeping until next cycle.")
        time.sleep(interval_hours * 3600)

continuous_monitoring(interval_hours=24)

这种方法使模型能够持续学习和适应，在处理新数据的过程中不断改进——这是在真实世界中保持高质量表现的一项关键要求。

## 4. 更进一步

### a. 模型评估

现在我们有了一个完全自动化的循环：它借助**评估**改进提示词，并在评分超过设定阈值时采纳新的提示词。

在生产环境中，你可以使用类似的框架，在新的用户请求到来时监控智能体的性能。
如上所述，这是一个简化的示例；在真实场景中，你需要增加额外的护栏，并采用人在回路（human-in-the-loop）的方式来批准新的提示词。

进一步延伸这一思路，我们还可以使用评估来测试不同的模型参数候选，例如模型版本、详细程度（verbosity）和推理强度。要查看可供考虑的完整参数集，请参阅 [Agents SDK 中的 ModelSettings 类](https://openai.github.io/openai-agents-python/ref/model_settings/#agents.model_settings.ModelSettings)

`compare_model_candidates` 函数示例展示了如何：
1. 优化提示词
2. 使用两个或更多不同的模型，基于优化后的提示词生成候选输出
3. 使用评估为候选输出打分，并选出最佳候选

只需少量重构，就可以将它整合到 `self_evolving_loop` 函数中。

> **注意：** 在生产环境中测试模型版本时，应限于同一系列内的版本（例如 gpt-5、gpt-5-mini、gpt-5-nano）。建议在生产部署之前完成跨系列的版本选择。


以下是加入模型比较代码后的最终 `self_evolving_loop`：

In [ ]:
from agents import Agent, Runner

async def eval_agent_candidate(agent: Agent, section: str, prompt_text: str, model_name: str):
    summary_result = await Runner.run(agent, section)
    summary = summary_result.final_output

    scores = await get_eval_grader_score(
        eval_id=EVAL_ID, summary=summary, section=section
    )
    average = calculate_grader_score(scores)
    lenient_passed = is_lenient_pass(scores, average)
    passed = all(entry.get("passed") is True for entry in scores)

    update_best_candidate(
        average_score=average,
        prompt_text=prompt_text,
        model_name=model_name,
        summary_text=summary,
        metadata={
            "section": section,
            "average_score": average,
            "grader_results": scores,
        },
        lenient_passed=lenient_passed,
    )

    return {"summary": summary, "scores": scores, "average": average, "passed": passed}

async def compare_model_candidates(
    summarization_prompt,
    eval_feedback: str,
    section: str,
    summary: str,
    model_candidates=None,
):
    """Improve the prompt, evaluate it across candidate models, and adopt the top performer."""
    if model_candidates is None:
        model_candidates = ["gpt-5", "gpt-5-mini"]

    metaprompt_result = await Runner.run(
        metaprompt_agent,
        input=METAPROMPT_TEMPLATE.format(
            original_prompt=summarization_prompt.current().prompt,
            section=section,
            summary=summary,
            reasoning=eval_feedback,
        ),
    )
    improved_prompt = metaprompt_result.final_output

    async def evaluate_model(model_name: str):
        candidate_agent = Agent(
            name=f"SummarizationAgent:{model_name}",
            instructions=improved_prompt,
            model=model_name,
        )
        result = await eval_agent_candidate(candidate_agent, section, improved_prompt, model_name)
        return model_name, candidate_agent, result

    best = {
        "average": float("-inf"),
        "passed": False,
        "agent": None,
        "model": None,
        "summary": None,
    }

    tasks = [asyncio.create_task(evaluate_model(model_name)) for model_name in model_candidates]
    for task in asyncio.as_completed(tasks):
        model_name, candidate_agent, result = await task
        print(
            f"Candidate average — {model_name}: {result['average']:.4f} "
            f"(passed={result.get('passed', False)})"
        )
        if result["average"] > best["average"]:
            best.update(
                {
                    "average": result["average"],
                    "model": model_name,
                    "summary": result.get("summary"),
                    "agent": candidate_agent,
                    "passed": result.get("passed", False),
                }
            )

    for task in tasks:
        if not task.done():
            task.cancel()

    if best["passed"] and best["model"]:
        summarization_prompt.update(
            new_prompt=improved_prompt,
            model=best["model"],
            metadata={"section": section, "summary": best["summary"]},
        )
        print(f"Updated summarization_prompt with passing model: {best['model']}")
        return make_summarization_agent(summarization_prompt.current())

    print(
        f"No passing models. Best candidate (model={best['model']}, "
        f"avg={best['average']:.4f}) did not pass. Prompt not updated."
    )
    return None

async def self_evolving_loop_with_model_comparison(summarization_agent: Agent) -> Agent:
    print(
        f"Starting self-evolving loop | Initial prompt v{summarization_prompt.current().version}"
    )
    print(f"Prompt: {summarization_prompt.current().prompt}")
    print(f"Model: {summarization_prompt.current().model}")
    print("-" * 80)

    reset_best_trackers()
    df = pd.read_csv("data/dataset.csv")

    with trace("Self-evolving Optimization Workflow: model comparison"):
        for _, row in df.head(5).iterrows():
            content = row.get("content")
            if pd.isna(content) or (isinstance(content, str) and not content.strip()):
                continue

            section_number = str(row["section_number"])
            section = str(content)
            current_version = summarization_prompt.current().version

            print(f"[Section {section_number}] Using prompt v{current_version}")

            summary_passed = False

            for attempt in range(1, MAX_OPTIMIZATION_RETRIES + 1):
                print(f"\tAttempt {attempt}: evaluating summary...")

                summary_result = await Runner.run(summarization_agent, section)
                summary = summary_result.final_output

                grader_scores = await get_eval_grader_score(
                    eval_id=EVAL_ID, summary=summary, section=section
                )
                average_score = calculate_grader_score(grader_scores)
                total_score = calculate_total_grader_score(grader_scores)
                lenient_passed = is_lenient_pass(grader_scores, average_score)
                print(
                    f"\tScores — avg={average_score:.3f}, total={total_score:.3f}, lenient_passed={lenient_passed}"
                )

                record_aggregate_prompt_score(
                    prompt_version=summarization_prompt.current().version,
                    prompt_text=summarization_prompt.current().prompt,
                    model_name=summarization_prompt.current().model,
                    average_score=average_score,
                    total_score=total_score,
                )

                update_best_candidate(
                    average_score=average_score,
                    total_score=total_score,
                    prompt_text=summarization_prompt.current().prompt,
                    model_name=summarization_prompt.current().model,
                    summary_text=summary,
                    metadata={
                        "section": section_number,
                        "average_score": average_score,
                        "grader_results": grader_scores,
                        "prompt_version": summarization_prompt.current().version,
                    },
                    lenient_passed=lenient_passed,
                    prompt_version=summarization_prompt.current().version,
                )

                if lenient_passed:
                    summary_passed = True
                    print(
                        f"\tPassed with prompt v{summarization_prompt.current().version} (model={summarization_prompt.current().model})"
                    )
                    break

                print("\tFailed eval. Improving prompt...")
                eval_feedback = collect_grader_feedback(grader_scores)

                new_agent = await compare_model_candidates(
                    summarization_prompt=summarization_prompt,
                    eval_feedback=eval_feedback,
                    section=section,
                    summary=summary,
                    # model_candidates could be given as an argument if you want to expand options.
                )

                if new_agent is None:
                    print(
                        "\tNo passing model found. Optimization failed for this section."
                    )
                    summary_passed = False
                else:
                    summarization_agent = new_agent
                    summary_passed = True
                    print(
                        f"\tPrompt improved → v{summarization_prompt.current().version} "
                        f"(model={summarization_prompt.current().model})"
                    )
                    break

            if not summary_passed:
                print(
                    "\tAll attempts failed; keeping latest prompt version "
                    f"v{summarization_prompt.current().version} (model={summarization_prompt.current().model}) for the next section."
                )

    summarization_agent = apply_best_candidate_if_needed()

    print("" + "-" * 80)
    print("Completed optimization loop.")
    print(f"Final prompt version: v{summarization_prompt.current().version}")
    print(f"Final model: {summarization_prompt.current().model}")
    aggregate_best = select_best_aggregate_prompt()
    if best_candidate["score"] > float("-inf"):
        print(
            f"Best lenient prompt: v{best_candidate.get('version')} (avg={best_candidate['score']:.3f}, model={best_candidate.get('model', 'unknown')})"
        )
    if aggregate_best:
        per_section = (
            aggregate_best.get("total_average", 0.0) / aggregate_best.get("count", 1)
            if aggregate_best.get("count")
            else 0.0
        )
        print(
            f"Aggregate best prompt: v{aggregate_best.get('version')} "
            f"(total={aggregate_best.get('total_score', 0.0):.3f}, avg/section={per_section:.3f}, model={aggregate_best.get('model', 'unknown')})"
        )
    print(f"Final prompt: {summarization_prompt.current().prompt}")
    print(f"Final model: {summarization_prompt.current().model}")
    return summarization_agent

summarization_agent = await self_evolving_loop_with_model_comparison(summarization_agent)


可以看到，输出与之前非常相似，只是多了关于各模型版本分数的信息：

### b. 使用 Genetic-Pareto（GEPA）优化提示词

我们已经证明了自我进化循环是有效的，并且可以借助 Evals 自主改进提示词。不过，我们改进系统提示词时依赖的是一个相对简单的静态元提示。在本节中，我们将探索一种更动态、更具反思性的方法：使用 Genetic-Pareto（GEPA）[[1]](#参考文献)——这是一个框架，它会采样智能体的执行轨迹，用自然语言对其进行反思，提出提示词修订方案，并通过迭代反馈循环让系统不断进化。

GEPA 方法在[这篇](https://doi.org/10.48550/arXiv.2507.19457)论文中有所描述，它为持续、自我改进的提示词优化提供了一个引人注目的蓝图。下面的代码大量借鉴了[这里](https://github.com/gepa-ai/gepa)的 GEPA Github 仓库。

In [ ]:
import pandas as pd
import gepa
from gepa import EvaluationBatch

# Extract sections from dataset
def read_csv_content(file_path: str) -> list[dict]:
    """Read csv and return section to summarize."""
    df = pd.read_csv(file_path)
    return [{'content': content} for content in df['content'].tolist()]

# Split dataset into training and validation sets
trainset = read_csv_content("data/dataset.csv")
val_cut = max(1, int(0.1 * len(trainset)))
valset = trainset[:val_cut] if len(trainset) > 1 else trainset

我们将通过添加一个小型适配器来复用之前的评分器和辅助函数，使我们的设置能够与 GEPA 配合使用。GEPA 的 `GEPAAdapter` 让接入我们的评估框架变得很容易。我们定义了三个钩子：
- `evaluate`：运行摘要，并使用上一节定义的评分器（即 chemical_name_grader、word_length_deviation_grader、cosine_similarity、llm_as_judge）进行评分。
- `get_components_to_update`：获取 GEPA 应当进化的文本字段（这里是 system_prompt）。
- `make_reflective_dataset`：将输入、输出和反馈打包，供反思使用。

In [ ]:
class EvalsBackedSummarizationAdapter:
    """
    Minimal adapter for GEPA:
      - evaluate(...) -> EvaluationBatch (scores + outputs + feedback-rich trajectories)
      - get_components_to_update(...) returns the prompt to update
      - make_reflective_dataset(...) packages examples for reflection
    """
    propose_new_texts = None  # use GEPA's default reflection flow

    def __init__(self, client, eval_id: str, gen_model: str = "gpt-5", user_prefix: str | None = None):
        self.client = client
        self.eval_id = eval_id
        self.gen_model = gen_model
        self.user_prefix = user_prefix or "Summarize:\n\n"

    # Same summarization agent as in the previous section
    def _summarize(self, system_prompt: str, section: str) -> str:
        resp = self.client.chat.completions.create(
            model=self.gen_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"{self.user_prefix}{section}"},
            ],
        )
        return resp.choices[0].message.content.strip()

    # Required by GEPA: run eval minibatch
    def evaluate(self, inputs: list[dict], candidate: dict, capture_traces: bool = True) -> EvaluationBatch:
        system_prompt = candidate["system_prompt"]

        scores: list[float] = []
        outputs: list[str] = []
        trajectories: list[dict] = []

        for item in inputs:
            section = item["content"]

            # 1) Generate with the candidate prompt
            summary = self._summarize(system_prompt, section)
            outputs.append(summary)

            # 2) Grade using previous evals pipeline
            run = run_eval(eval_id=self.eval_id, section=section, summary=summary)
            out_items = poll_eval_run(eval_id=self.eval_id, run_id=run.id)
            grader_scores = parse_eval_run_output(out_items)

            # 3) Score + actionable feedback
            scalar = calculate_grader_score(grader_scores)
            feedback = collect_grader_feedback(grader_scores) or "All graders passed; keep precision and coverage."

            scores.append(float(scalar))
            trajectories.append(
                {
                    "inputs": {"section": section},
                    "generated_output": summary,
                    "metrics": {
                        "combined": float(scalar),
                        "by_grader": grader_scores,  # keeping for analysis if needed
                    },
                    "feedback": feedback,
                }
            )

        return EvaluationBatch(scores=scores, outputs=outputs, trajectories=trajectories)

    # Required by GEPA: text field to evolve
    def get_components_to_update(self, candidate: dict) -> list[str]:
        return ["system_prompt"]

    # Required by GEPA: build the reflective dataset the reflection LM will read
    def make_reflective_dataset(self, candidate: dict, eval_batch: EvaluationBatch, components_to_update: list[str]) -> dict:
        examples = []
        for traj in (eval_batch.trajectories or []):
            examples.append(
                {
                    "Inputs": {"section": traj["inputs"]["section"]},
                    "Generated Outputs": traj["generated_output"],
                    "Feedback": traj["feedback"],
                }
            )
        return {"system_prompt": examples}

适配器准备好后，我们就可以运行 GEPA 了。为了便于比较，我们使用与之前自我进化循环相同的初始提示词（`"You are a summarization assistant. Given a section of text, produce a summary."`）和模型（这里是 `gpt-5`）。我们将适配器实例、种子候选以及训练集/验证集传给 `gepa.optimize(...)`。在优化过程中，GEPA 会反复调用适配器为候选打分、对反馈进行反思，并最终生成进化得最好的提示词。

_注意：GEPA 可能需要约 10-15 分钟才能完成。_

In [ ]:
seed_candidate = {"system_prompt": "You are a summarization assistant. Given a section of text, produce a summary."}

adapter = EvalsBackedSummarizationAdapter(
    client=client,
    eval_id=EVAL_ID,
    gen_model=summarization_prompt.current().model,  
)

# Keeping max_metric_calls small for the cookbook. 
# In practice, use a larger value to allow more optimization iterations.
result = gepa.optimize(
    seed_candidate=seed_candidate,
    trainset=trainset,
    valset=valset,
    adapter=adapter,
    reflection_lm="gpt-5",
    max_metric_calls=10,
    track_best_outputs=True,
    display_progress_bar=True
)

best_prompt = result.best_candidate["system_prompt"]
print("\n=== Best evolved instruction ===\n")
print(best_prompt)

下面是上述代码的一个示例（节选）输出：

在本 cookbook 中，我们探索了三种不同的提示词优化方法：

- **OpenAI 平台优化器：** 使用 _Optimize_ 按钮，结合包含人工录入反馈（点赞/点踩和文字评论）的数据集，我们只需极少的配置就快速得到了一个很强的提示词。这种方法擅长快速迭代，但无法提供生产环境所需的自动化能力。

- **使用静态元提示进行优化：** 我们的循环结合了四种不同的评分器，无需人工干预即可实现自动探索和迭代式自我改进。然而，它的探索空间受限于单一的静态元提示，而且评估是逐个章节进行的。因此，这种方法存在过度拟合即时评分器反馈、而无法实现更广泛泛化的风险。

- **GEPA 优化：** 它提供了更结构化的搜索过程：反思式更新同时参考定量分数和文字反馈，候选在一个数据集上训练、在另一个数据集上验证。这种方法产生了更稳健、泛化能力更强的提示词，并为其性能提供了更清晰的实证依据。

_注意：附录中提供了每种方法所生成提示词的示例。_

根据你的用例，你可以优先考虑速度（OpenAI 优化器）、轻量级自动化（静态元提示）或系统化的泛化能力（GEPA）。在实践中，将这些方法结合起来——从快速迭代开始，逐步过渡到反思式优化——可以同时兼顾敏捷性和性能。

祝编码愉快！

## 贡献者

本 cookbook 由 [Bain](https://www.bain.com) 与 [OpenAI](https://openai.com) 联合合作完成。

[Calvin Maguranis](https://www.linkedin.com/in/calvin-maguranis-b9956045/)  
[Fanny Perraudeau](https://www.linkedin.com/in/fanny-sabran-perraudeau-494b7573/)   
[Giorgio Saladino](https://www.linkedin.com/in/giorgio-saladino-202/)   
[Shikhar Kwatra](https://www.linkedin.com/in/shikharkwatra/)    
[Valentina Frenkel](https://www.linkedin.com/in/valentina-frenkel/)  

## 参考文献

[1] _GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning_，作者：Lakshya A Agrawal, Shangyin Tan, Dilara Soylu, Noah Ziems, Rishi Khare, Krista Opsahl-Ong, Arnav Singhvi, Herumb Shandilya, Michael J Ryan, Meng Jiang, Christopher Potts, Koushik Sen, Alexandros G. Dimakis, Ion Stoica, Dan Klein, Matei Zaharia, Omar Khattab - https://arxiv.org/abs/2507.19457

## 附录

### 输出提示词示例：

- **初始提示词：**  
```pgsql 
You are a summarization assistant. Given a section of text, produce a summary.
```

- **OpenAI 平台优化器：** 
```pgsql 
You are a summarization assistant.
Task: Summarize the provided text concisely and accurately.
Output requirements:
- Output only the summary. Do not add titles, labels (e.g.,
"Summary:"), prefaces, or commentary.
- Preserve the document's structure. If multiple sections/subsections appear, summarize each one.
- Use a numbered list for sections/subsections (use their numbers/titles when present).
- Under each, use short dash bullets for key points.
- If there is only a single short section, return a brief bullet list or 1-2 concise sentences.
- Split any inline lists into separate bullets.
- Use plain, simple language. Keep bullets tight (ideally one line each). Remove redundancy.
- Include important quantitative details (values, units, conditions) and constraints. Do not invent information.
- Keep formatting simple: plain text, "1." numbering and "-" bullets only. No tables or special markup.
- Retain exact technical terms/notation from the source (e.g., chemical names, isotopic labels).
- If a section is explicitly marked "Not applicable," include that status; otherwise do not add it.
```

- **静态元提示：** 
```pgsql 
You are a technical summarization assistant for scientific and regulatory documentation. Your task is to generate a concise, comprehensive, and fully detailed summary of any scientific, technical, or regulatory text provided. Strictly adhere to the following instructions:

---

**1. Complete and Exact Information Inclusion**  
- Capture *every* explicit fact, technical value, specification, quantity, measurement, regulatory reference, entity, process, site, and contextual detail verbatim from the source text.
- Do not omit or generalize any explicit information, no matter how minor.

**2. Precise Terminology and Named Entity Retention**  
- Reproduce all names of chemicals, drugs, mixtures, buffer components, devices, companies, institutions, regulatory standards, section numbers, and procedural labels *exactly as stated*.
- Report all quantities, measurements, concentrations, ratios, masses, volumes, compositions, pH values, and units precisely as given.
- Do not paraphrase, rename, substitute, or simplify any term or value.

**3. All Procedural Details and Justifications**  
- Explicitly include all described procedures, technical processes (e.g., terminal sterilization, aseptic processing), operational constraints, process justifications, compliance requirements, and standards references.
- Clearly state all reasons provided for choosing or omitting particular methods or processes.

**4. Regulatory and Compliance References**  
- Accurately cite all regulations, standards (e.g., USP <797>), compliance statements, section numbers, and cross-references as in the original.
- Include all explicit mentions of compliance, applicability, and site location details.

**5. Explicit Statements of Absence, Limitations, and Applicability**  
- Clearly state any declarations of absence, inapplicability (“Not applicable”), or limitations exactly as written in the source.

**6. Structural and Organizational Fidelity**  
- Precisely reflect the original document’s section and subsection hierarchy, using clear section labels and indentation.
- Present all enumerations, lists, and tabulated data in structured bullet-point or numbered format, organized in accordance with the source document’s arrangement.

**7. No Paraphrasing, Summarizing, or Reinterpretation**  
- Do *not* paraphrase, summarize contextually, reinterpret, or alter the meaning or sequence of any content.
- Remove only literal repetitions or redundant phrasing; otherwise, preserve all explicit statements, technical details, and contextual notes.

---

**Summary Output Objective:**  
Produce a summary that delivers the full technical, factual, and regulatory content and structure of the original text, reformatted by eliminating only redundant language. The summary must enable audit, regulatory review, or peer reference without loss of any explicit information or terminology from the source.

---

*Apply these instructions rigorously to every provided document section to ensure scientific and regulatory accuracy and completeness.*
```

- **GEPA 优化器**： 
```pgsql 
You are a domain-aware summarization assistant for technical pharmaceutical texts. Given a “section” of text, produce a concise, single-paragraph summary that preserves key technical facts and exact nomenclature.

Length and format
- Write 1–3 sentences totaling about 45–70 words (target ~60; never exceed 90).
- Use one paragraph; no bullets, headings, tables, or heavy formatting.

Exact names and notation
- Include every chemical name that appears in the section at least once, using the exact original spelling, capitalization, punctuation, isotopic labels, brackets, hyphens, salts, buffer names, and parenthetical qualifiers. Treat distinct case/format variants as distinct names (e.g., [1-13C]pyruvic acid and [1-13C]Pyruvic acid are separate and each must appear once).
- Examples you must preserve verbatim when present: Hyperpolarized Pyruvate (13C) Injection; non-polarized Pyruvate Injection; Pyruvate (13C) Injection; hyperpolarized [1-13C]pyruvate; Mixture of [1-13C]pyruvic acid and 15 mM AH111501 sodium salt; TRIS/EDTA buffer solution; TRIS; NaOH; Na2EDTA; [1-13C]pyruvic acid; AH111501 sodium salt.
- Also preserve exact study identifiers, batch codes, section numbers, regulatory citations, and instrument parameters as written (e.g., GE-101-001, GE-101-003, USP <797>, 3.2.P.5.2.5, FFF106/140-806, FFF106/142-806, 3T MRI, 5 degree RF pulse, TR=3s, 90 degree pulse, 64 averages, TR=10s, 10 μl Gd/ml solution).

Content prioritization (if space is tight)
1) What the section is about (topic/purpose).
2) All named chemical entities and compositions (list all chemical names at least once; include concentrations/amounts if given).
3) Critical process/handling facts (e.g., aseptic processing vs terminal sterilization; ISO classifications; filtration specs; compounding/filling steps; temperatures/times/volumes; storage/administration limits).
4) Container/packaging specifics (e.g., cryovials, “sterile fluid path”).
5) Microbiological/testing/regulatory details (e.g., sterility/pyrogenicity testing timing; USP <797>; state board compliance; site/manufacturer if stated).
6) Overages/single-dose formulas and key quantities.

Numerical fidelity
- Preserve all critical numbers and units exactly (e.g., 1.44 g, 27.7 mg, 15 mM, 18 mL, 1.47 g, two 0.2 μm filters, ISO 7, ISO 5, 38 mL).
- Include testing/analysis parameters when present (e.g., polarization/relaxation time (T1); number of spectra; pulse angles; TR values; MRI location relative to clean room).

Style and compression
- Be neutral and factual; do not infer unstated information.
- Consolidate repeated statements; compress lists with commas/semicolons to save words.
- Mention tables/figures only to convey key data; do not reproduce them.
- If many chemicals are present, ensure each distinct name appears once; group them succinctly.
- Avoid symbols or special formatting not in the source text.

Common domain cues to include when present
- Aseptic processing vs terminal sterilization and the rationale/timing (e.g., “tested for sterility and pyrogenicity subsequent to patient administration”).
- Environmental/processing controls (ISO 7/ISO 5; LAF unit; filtration; filling/weight targets per cryovial).
- Site/regulatory context (e.g., USP <797>; California State Board of Pharmacy; University of California, San Francisco Department of Clinical Pharmacy).
- Study/kit equivalence statements (e.g., equivalence to GE-101-001/GE-101-003 formulations).
- QC/measurement methods (e.g., capacitive threshold at Administration syringe nominal 38 mL).

Self-check before finalizing
- Does the paragraph contain every distinct chemical name exactly as written in the section (including case and notation variants)?
- Is the summary 45–70 words (≤90), in a single paragraph?
- Are the most critical process/regulatory/testing details and all key numbers preserved without unnecessary verbosity?`
```